In [91]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
    
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.preprocessing import PowerTransformer

In [92]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [93]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [94]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [95]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

In [96]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

### COX assumption in Train data

In [97]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic      p  -log2(p)
MTV                       0.07   0.80      0.33
SUVpeak                   0.17   0.68      0.56
TLG                       0.38   0.54      0.90
age                       0.09   0.77      0.38
cavum_oris                0.00   0.99      0.02
charlson                  1.66   0.20      2.34
female                    2.67   0.10      3.29
histgrade_high            0.13   0.72      0.48
hpv_related              12.67 <0.005     11.39
hypopharynx               0.00   0.99      0.01
larynx                    0.00   0.98      0.03
oropharynx                0.00   0.98      0.02
pack_years                0.06   0.81      0.31
uicc8_III-IV              

In [98]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


In [99]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='OS', event_col='event_OS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 82 right-censored observations>
         test_name = proportional_hazard_test

---
                test_statistic    p  -log2(p)
MTV                       0.59 0.44      1.17
SUVpeak                   0.01 0.91      0.14
TLG                       0.00 0.96      0.06
age                       0.43 0.51      0.97
cavum_oris                0.41 0.52      0.94
charlson                  0.87 0.35      1.51
female                    2.39 0.12      3.04
histgrade_high            0.11 0.74      0.44
hpv_related               6.46 0.01      6.51
hypopharynx               0.04 0.84      0.25
larynx                    0.97 0.33      1.62
oropharynx                0.85 0.36      1.49
pack_years                0.00 0.99      0.01
uicc8_III-IV              0.00 0.95      0.07


In [100]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['hpv_related'], dtype='object')


###### penalizer values essentially result in same result 

## Test dataset: MAASTRO 

In [101]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [102]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [103]:
# need to choose patient_id from MAASTRO_D1 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [104]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event
0,1,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342,44.43,1.0
3,4,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979,37.20,1.0
4,6,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782,19.00,1.0
95,111,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492,58.93,0.0


In [105]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [106]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event


In [107]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [108]:
# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [109]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 14)
y_train:  (139,)


In [110]:
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [111]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 16)

In [112]:
# VIF dataframe 
vif_data = pd.DataFrame() 
vif_data["feature"] = X.columns 
  
# calculating VIF for each feature 
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                          for i in range(len(X.columns))] 

vif_data

,feature,VIF
0,age,1.136852
1,female,1.197250
2,cavum_oris,7.993011
3,oropharynx,60.199385
4,hypopharynx,11.118710
5,larynx,14.183474
6,histgrade_high,1.150479
7,hpv_related,4.309385
8,charlson,1.302350
9,pack_years,1.647441


# Yeo-Johnson Transformation

In [113]:
original_X = X.copy()

In [114]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

## Standardize the data but not the categorical columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

### Separate the categorical and non-categorical columns
X_categorical = X[categorical_columns]
X_numeric = X.drop(categorical_columns, axis=1)
X_numeric_columns = X_numeric.columns
X_numeric_index = X_numeric.index

### Standardize non-categorical and then concat with the categorical
pt = PowerTransformer(method='yeo-johnson')
X_numeric_std = pt.fit_transform(X_numeric)
X_numeric_std = pd.DataFrame(X_numeric_std, columns=X_numeric_columns, index=X_numeric_index)
X_std = pd.concat([X_categorical, X_numeric_std], axis=1)

## Sort the order of the columns as it was in the clinical train 
X_std = X_std[original_X.columns]

In [117]:
# Standardize X_MAASTRO 
MAASTRO_new = X_MAASTRO.copy()
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = pt.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[X_new.columns]

In [118]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = MAASTRO_new_std 

In [119]:
X_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,54.238356,1,0,1,0,0,1,0.0,0,0.000000,0.0,14.473272,7.934,86.228420
1,54.539726,0,0,0,0,1,0,0.0,1,27.404795,0.0,5.044678,1.656,7.040100
2,59.019178,0,1,0,0,0,1,0.0,1,41.019178,1.0,7.839043,14.502,83.569669
3,70.726027,0,0,0,0,1,0,0.0,1,37.500000,0.0,2.880631,2.440,5.567091
4,67.865753,0,0,0,0,1,0,0.0,1,53.000000,0.0,5.402006,3.668,16.150550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,60.435616,0,0,1,0,0,1,1.0,0,0.000000,0.0,9.290139,3.650,26.280140
135,68.794521,0,0,1,0,0,1,1.0,0,0.000000,1.0,7.172883,18.967,101.754834
136,57.498630,0,0,1,0,0,1,1.0,1,39.498630,0.0,13.873187,6.370,66.273201
137,65.684932,0,0,1,0,0,1,1.0,1,71.527397,1.0,7.507419,12.443,71.832443


In [120]:
X_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,-0.785036,1,0,1,0,0,1,0.0,0,-1.533345,0.0,0.750782,0.080033,0.343545
1,-0.746998,0,0,0,0,1,0,0.0,1,0.397233,0.0,-1.213660,-1.722060,-1.657833
2,-0.175256,0,1,0,0,0,1,0.0,1,0.830437,1.0,-0.474112,0.735716,0.318631
3,1.371631,0,0,0,0,1,0,0.0,1,0.727938,0.0,-1.992620,-1.291020,-1.835650
4,0.987008,0,0,0,0,1,0,0.0,1,1.144253,0.0,-1.106304,-0.816679,-1.003396
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,0.007949,0,0,1,0,0,1,1.0,0,-1.533345,0.0,-0.158214,-0.822461,-0.611074
135,1.111443,0,0,1,0,0,1,1.0,0,-1.533345,1.0,-0.632361,1.008070,0.474941
136,-0.370651,0,0,1,0,0,1,1.0,1,0.786825,0.0,0.658564,-0.170908,0.133612
137,0.696583,0,0,1,0,0,1,1.0,1,1.554125,1.0,-0.551725,0.574668,0.197985


In [121]:
MAASTRO_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623
1,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700
2,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342
3,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979
4,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782
95,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868
96,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274
97,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492


In [122]:
MAASTRO_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG
0,-0.688798,0,0,1,0,0,1,1,1,-1.533345,0,0.893634,1.188930,1.218887
1,-0.688798,0,0,1,0,0,0,0,0,0.104884,1,-0.254678,-0.307894,-0.335276
2,-0.688798,0,0,1,0,0,0,0,1,-0.713198,1,0.595996,0.059450,0.228548
3,0.081262,1,0,0,0,1,1,0,1,0.940197,1,-0.296877,0.076321,-0.145668
4,1.273609,0,0,1,0,0,1,1,1,1.285347,0,-0.058378,0.786849,0.519421
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,0.738387,1,0,0,0,1,0,0,0,1.192311,1,2.645651,-0.219083,0.751363
95,0.342484,0,0,0,0,1,0,0,1,3.094727,1,0.526028,-0.033134,0.168333
96,0.342484,0,0,1,0,0,1,1,1,-1.533345,1,-0.230255,0.867254,0.481447
97,-0.815082,0,0,1,0,0,1,1,0,-1.533345,0,0.706707,0.336049,0.486335


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [123]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 23:58:35,307] A new study created in memory with name: no-name-20cde77c-d0ac-487e-9ad3-2666b073c79c


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.7142857142857143
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.8354430379746836


[I 2024-04-13 23:58:37,908] A new study created in memory with name: no-name-833a1218-af08-45a3-969c-a67341914f73


Fold 5 C-index: 0.6431924882629108
[I 2024-04-13 23:58:37,807] Trial 0 finished with value: 0.7434989412546489 and parameters: {}. Best is trial 0 with value: 0.7434989412546489.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7434989412546489], datetime_start=datetime.datetime(2024, 4, 13, 23, 58, 35, 498842), datetime_complete=datetime.datetime(2024, 4, 13, 23, 58, 37, 805906), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7434989412546489


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.16883035103807653
Fold 2 IBS: 0.19635745123057224
Fold 3 IBS: 0.16550671851920126
Fold 4 IBS: 0.14957197387797055
Fold 5 IBS: 0.26973255955213016
[I 2024-04-13 23:58:40,087] Trial 0 finished with value: 0.18999981084359013 and parameters: {}. Best is trial 0 with value: 0.18999981084359013.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.18999981084359013], datetime_start=datetime.datetime(2024, 4, 13, 23, 58, 38, 113625), datetime_complete=datetime.datetime(2024, 4, 13, 23, 58, 40, 85451), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.18999981084359013


In [124]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [125]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.743
train_ibs:  0.19


#### Test

In [126]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [127]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.581
IBS score: 0.281


In [128]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [129]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [130]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:58:41,005] A new study created in memory with name: no-name-88dc9f68-b041-4a39-b943-aea865c0a3ba


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6298701298701299
Fold 2 C-index: 0.640625
Fold 3 C-index: 0.7524509803921569
Fold 4 C-index: 0.7784810126582279
Fold 5 C-index: 0.5727699530516432
[I 2024-04-13 23:58:42,397] Trial 0 finished with value: 0.6748394151944316 and parameters: {}. Best is trial 0 with value: 0.6748394151944316.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6748394151944316], datetime_start=datetime.datetime(2024, 4, 13, 23, 58, 41, 72039), datetime_complete=datetime.datetime(2024, 4, 13, 23, 58, 42, 395754), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6748394151944316


[I 2024-04-13 23:58:42,448] A new study created in memory with name: no-name-23b7025d-bcc6-4d91-a1a3-ce953f49398c


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651730092662
Fold 2 IBS: 0.22157791055888598
Fold 3 IBS: 0.20453594148397225
Fold 4 IBS: 0.22473803361997696
Fold 5 IBS: 0.21812431365411103
[I 2024-04-13 23:58:44,519] Trial 0 finished with value: 0.21659054332357458 and parameters: {}. Best is trial 0 with value: 0.21659054332357458.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054332357458], datetime_start=datetime.datetime(2024, 4, 13, 23, 58, 42, 669817), datetime_complete=datetime.datetime(2024, 4, 13, 23, 58, 44, 519110), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054332357458


In [131]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [132]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.675
train_ibs:  0.217


#### Test

In [133]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [134]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.534


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [135]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [136]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:58:45,635] A new study created in memory with name: no-name-7bb41429-d3e0-45a2-8aec-0833823df13d


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.8354430379746836


[I 2024-04-13 23:58:50,015] A new study created in memory with name: no-name-8534428e-cc16-49bc-8cb6-1eafb604028f


Fold 5 C-index: 0.6572769953051644
[I 2024-04-13 23:58:49,919] Trial 0 finished with value: 0.746490912691111 and parameters: {}. Best is trial 0 with value: 0.746490912691111.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.746490912691111], datetime_start=datetime.datetime(2024, 4, 13, 23, 58, 45, 690582), datetime_complete=datetime.datetime(2024, 4, 13, 23, 58, 49, 905588), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.746490912691111


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.1693166643871831
Fold 2 IBS: 0.19627569923765678
Fold 3 IBS: 0.16332206774428829
Fold 4 IBS: 0.14781384832535083
Fold 5 IBS: 0.26849204574549806
[I 2024-04-13 23:58:53,643] Trial 0 finished with value: 0.18904406508799543 and parameters: {}. Best is trial 0 with value: 0.18904406508799543.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.18904406508799543], datetime_start=datetime.datetime(2024, 4, 13, 23, 58, 50, 208696), datetime_complete=datetime.datetime(2024, 4, 13, 23, 58, 53, 631158), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.18904406508799543


In [137]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [138]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.746
train_ibs:  0.189


#### Test

In [139]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [140]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.592


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.28


In [141]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [142]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:58:56,340] A new study created in memory with name: no-name-fccab477-810c-4229-af26-ce38ac6a6712


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.6572769953051644
[I 2024-04-13 23:58:59,758] Trial 0 finished with value: 0.746490912691111 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.746490912691111.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6572769953051644
[I 2024-04-13 23:59:05,352] Trial 1 finished with value: 0.7456470308345708 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.746490912691111.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6572769953051644
[I 2024-04-13 23:59:09,701] Trial 2 finished with value: 0.7456470308345708 and parameters: {'l1_ratio': 0.22692876841884668}. Be

Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.6572769953051644
[I 2024-04-14 00:00:42,878] Trial 24 finished with value: 0.746490912691111 and parameters: {'l1_ratio': 0.8064824763528503}. Best is trial 0 with value: 0.746490912691111.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.6572769953051644
[I 2024-04-14 00:00:46,083] Trial 25 finished with value: 0.746490912691111 and parameters: {'l1_ratio': 0.6339676394936953}. Best is trial 0 with value: 0.746490912691111.
Fold 1 C-index: 0.6363636363636364
Fold 2 C-index: 0.6428571428571429
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.7721518987341772
Fold 5 C-index: 0.596244131455399
[I 2024-04-14 00:00:47,757] Trial 26 finished with value: 0.6814841461957966 and parameters: {'l1_ratio': 0.015423757551295547}. B

Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6572769953051644
[I 2024-04-14 00:01:56,144] Trial 48 finished with value: 0.7456470308345708 and parameters: {'l1_ratio': 0.45725440612299806}. Best is trial 0 with value: 0.746490912691111.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6572769953051644
[I 2024-04-14 00:01:59,209] Trial 49 finished with value: 0.7456470308345708 and parameters: {'l1_ratio': 0.3754056069137437}. Best is trial 0 with value: 0.746490912691111.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.7941176470588235
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6572769953051644
[I 2024-04-14 00:02:02,803] Trial 50 finished with value: 0.7456470308345708 and parameters: {'l1_ratio': 0.6064718647459368}.

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.16939356487827184
Fold 2 IBS: 0.19677633886148008
Fold 3 IBS: 0.1631953006442755
Fold 4 IBS: 0.14789851767436374
Fold 5 IBS: 0.2683969898141077
[I 2024-04-14 00:04:18,858] Trial 0 finished with value: 0.18913214237449974 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.18913214237449974.
Fold 1 IBS: 0.1694912710909019
Fold 2 IBS: 0.19770864903522367
Fold 3 IBS: 0.16461913118179491
Fold 4 IBS: 0.14831322413708167
Fold 5 IBS: 0.2682120259189454
[I 2024-04-14 00:04:21,906] Trial 1 finished with value: 0.18966886027278954 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.18913214237449974.
Fold 1 IBS: 0.16950118238298933
Fold 2 IBS: 0.19785719051161688
Fold 3 IBS: 0.1645624877332342
Fold 4 IBS: 0.14817270753131573
Fold 5 IBS: 0.268248614110171
[I 2024-04-14 00:04:25,219] Trial 2 finished with value: 0.1896684364538654 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.18913214237449974

Fold 1 IBS: 0.16933662501128705
Fold 2 IBS: 0.19650838285558786
Fold 3 IBS: 0.16335035126098837
Fold 4 IBS: 0.1477213551728845
Fold 5 IBS: 0.2684968762348328
[I 2024-04-14 00:05:23,181] Trial 25 finished with value: 0.18908271810711613 and parameters: {'l1_ratio': 0.9363009671094719}. Best is trial 6 with value: 0.18902370112339847.
Fold 1 IBS: 0.21329546612926484
Fold 2 IBS: 0.2206045919636927
Fold 3 IBS: 0.20367201451615735
Fold 4 IBS: 0.22360038965205914
Fold 5 IBS: 0.21789198128063986
[I 2024-04-14 00:05:24,363] Trial 26 finished with value: 0.2158128887083628 and parameters: {'l1_ratio': 0.015423757551295547}. Best is trial 6 with value: 0.18902370112339847.
Fold 1 IBS: 0.16940894356216168
Fold 2 IBS: 0.19695045885917128
Fold 3 IBS: 0.16300729331681135
Fold 4 IBS: 0.14853733095928068
Fold 5 IBS: 0.2684088437874255
[I 2024-04-14 00:05:27,024] Trial 27 finished with value: 0.1892625740969701 and parameters: {'l1_ratio': 0.6022588122254688}. Best is trial 6 with value: 0.189023701123

Fold 1 IBS: 0.1694905593580144
Fold 2 IBS: 0.19776818858044026
Fold 3 IBS: 0.16457935699733217
Fold 4 IBS: 0.14819660433157147
Fold 5 IBS: 0.268270109219041
[I 2024-04-14 00:06:26,096] Trial 50 finished with value: 0.18966096369727986 and parameters: {'l1_ratio': 0.2311229033161103}. Best is trial 31 with value: 0.18902199038509662.
Fold 1 IBS: 0.1693245407561358
Fold 2 IBS: 0.19627705473388177
Fold 3 IBS: 0.1632805799677891
Fold 4 IBS: 0.14776918362708977
Fold 5 IBS: 0.26846286888583126
[I 2024-04-14 00:06:28,950] Trial 51 finished with value: 0.18902284559414556 and parameters: {'l1_ratio': 0.9799805566834315}. Best is trial 31 with value: 0.18902199038509662.
Fold 1 IBS: 0.1693208013101448
Fold 2 IBS: 0.1964992102077115
Fold 3 IBS: 0.16326112633889883
Fold 4 IBS: 0.1477766982484385
Fold 5 IBS: 0.26854959082925056
[I 2024-04-14 00:06:31,216] Trial 52 finished with value: 0.18908148538688882 and parameters: {'l1_ratio': 0.973873594635656}. Best is trial 31 with value: 0.18902199038509

Fold 1 IBS: 0.16933159838976095
Fold 2 IBS: 0.1965064417315675
Fold 3 IBS: 0.1633851894024835
Fold 4 IBS: 0.14773848846386436
Fold 5 IBS: 0.2685131075754331
[I 2024-04-14 00:07:30,711] Trial 75 finished with value: 0.1890949651126219 and parameters: {'l1_ratio': 0.9477404366976027}. Best is trial 31 with value: 0.18902199038509662.
Fold 1 IBS: 0.16932473978852686
Fold 2 IBS: 0.19627717308437517
Fold 3 IBS: 0.16327905934000325
Fold 4 IBS: 0.14776847060934775
Fold 5 IBS: 0.2684628316037903
[I 2024-04-14 00:07:32,852] Trial 76 finished with value: 0.1890224548852087 and parameters: {'l1_ratio': 0.9794888852309296}. Best is trial 31 with value: 0.18902199038509662.
Fold 1 IBS: 0.1693231660203654
Fold 2 IBS: 0.19650074862145128
Fold 3 IBS: 0.16324258292199828
Fold 4 IBS: 0.14776834087788812
Fold 5 IBS: 0.2685416460331903
[I 2024-04-14 00:07:35,192] Trial 77 finished with value: 0.18907529689497868 and parameters: {'l1_ratio': 0.9680934252919277}. Best is trial 31 with value: 0.1890219903850

In [143]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [144]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.746
train_ibs:  0.189


#### Test

In [145]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [146]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6964995386793018)

test_cindex : 0.592


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.9775140863480035)

test_ibs:  0.28


In [147]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [148]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 00:08:28,796] A new study created in memory with name: no-name-0f6c0e91-4986-40ae-8192-e0a79d80e1b1


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.6339285714285714
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8544303797468354
Fold 5 C-index: 0.6408450704225352
[I 2024-04-14 00:08:43,198] Trial 0 finished with value: 0.7372744686944293 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7372744686944293.
Fold 1 C-index: 0.7012987012987013
Fold 2 C-index: 0.6875
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8607594936708861
Fold 5 C-index: 0.647887323943662
[I 2024-04-14 00:08:55,148] Trial 1 finished with value: 0.74027341750814 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_features': '

Fold 1 C-index: 0.70995670995671
Fold 4 C-index: 0.8755274261603375
Fold 5 C-index: 0.7230046948356808
[I 2024-04-14 00:10:49,774] Trial 16 finished with value: 0.7913196149300414 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 15, 'min_samples_leaf': 16, 'max_depth': 2, 'n_estimators': 62, 'oob_score': True, 'max_samples': 0.9622193970781673, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.07748782870655765, 'warm_start': True}. Best is trial 14 with value: 0.8012393138587386.
Fold 1 C-index: 0.6796536796536796
Fold 2 C-index: 0.8571428571428571
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.8839662447257384
Fold 5 C-index: 0.7417840375586855
[I 2024-04-14 00:12:49,039] Trial 17 finished with value: 0.7952544618554078 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 16, 'min_samples_leaf': 15, 'max_depth': 7, 'n_estimators': 127, 'oob_score': True, 'max_samples': 0.8172323984404596, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.1032735747299450

Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.8571428571428571
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.9071729957805907
Fold 5 C-index: 0.7981220657276995
[I 2024-04-14 00:32:00,120] Trial 31 finished with value: 0.8102976168343801 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 7, 'min_samples_leaf': 7, 'max_depth': 12, 'n_estimators': 317, 'oob_score': True, 'max_samples': 0.8965443067001324, 'max_features': None, 'min_weight_fraction_leaf': 0.04869831874272522, 'warm_start': True}. Best is trial 30 with value: 0.8141045941707695.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.8571428571428571
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.9029535864978903
Fold 5 C-index: 0.7746478873239436
[I 2024-04-14 00:32:10,746] Trial 32 finished with value: 0.8044151254239035 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 6, 'min_samples_leaf': 7, 'max_depth': 13, 'n_estimators': 317, 'oob_score': True, 'max_samples': 0.7822231835771817,

...

Fold 1 IBS: 0.21398563901141657
Fold 2 IBS: 0.22134596740550985
Fold 3 IBS: 0.2048025990579434
Fold 4 IBS: 0.22461415055237854
Fold 5 IBS: 0.21851803782809134
[I 2024-04-14 00:45:35,888] Trial 32 finished with value: 0.21665327877106794 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 18, 'n_estimators': 451, 'oob_score': False, 'max_samples': 0.7396444271446992, 'max_features': None, 'min_weight_fraction_leaf': 0.4501495537267681}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.19337121417130673
Fold 2 IBS: 0.19511882700454072
Fold 3 IBS: 0.18642967451701625
Fold 4 IBS: 0.1895345717679471
Fold 5 IBS: 0.21856358775020004
[I 2024-04-14 00:45:51,734] Trial 33 finished with value: 0.19660357504220216 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 12, 'min_samples_leaf': 1, 'max_depth': 19, 'n_estimators': 493, 'oob_score': False, 'max_samples': 0.9933327757837453, 'max_features': None, 'min_weight_fraction_leaf

Fold 1 IBS: 0.21398162469590376
Fold 2 IBS: 0.22136781149843104
Fold 3 IBS: 0.20480753036595514
Fold 4 IBS: 0.22461586717115936
Fold 5 IBS: 0.21858421480458282
[I 2024-04-14 00:49:25,350] Trial 48 finished with value: 0.21667140970720644 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 459, 'oob_score': False, 'max_samples': 0.8399357435042919, 'max_features': None, 'min_weight_fraction_leaf': 0.43949783817145754}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.18607291644289184
Fold 2 IBS: 0.1997166990883884
Fold 3 IBS: 0.17871196581310303
Fold 4 IBS: 0.1866755623669813
Fold 5 IBS: 0.21401906861191788
[I 2024-04-14 00:49:39,906] Trial 49 finished with value: 0.19303924246465648 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 6, 'min_samples_leaf': 20, 'max_depth': 6, 'n_estimators': 480, 'oob_score': False, 'max_samples': 0.9976678104279734, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 1 IBS: 0.17214163753608527
Fold 2 IBS: 0.19020363580055616
Fold 3 IBS: 0.16550959046326688
Fold 4 IBS: 0.16526794959392554
Fold 5 IBS: 0.22833087784482511
[I 2024-04-14 00:52:42,122] Trial 64 finished with value: 0.1842907382477318 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 7, 'max_depth': 18, 'n_estimators': 359, 'oob_score': False, 'max_samples': 0.8662053800926933, 'max_features': None, 'min_weight_fraction_leaf': 0.39542258978805284}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.2139792007492792
Fold 2 IBS: 0.22131668306329724
Fold 3 IBS: 0.20481655835559517
Fold 4 IBS: 0.22465389902940344
Fold 5 IBS: 0.2186107338002615
[I 2024-04-14 00:52:52,855] Trial 65 finished with value: 0.21667541499956733 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 12, 'min_samples_leaf': 8, 'max_depth': 20, 'n_estimators': 370, 'oob_score': False, 'max_samples': 0.8126605473414451, 'max_features': None, 'min_weight_fraction_leaf'

Fold 1 IBS: 0.21398666839515476
Fold 2 IBS: 0.22133561292920703
Fold 3 IBS: 0.20476084552868884
Fold 4 IBS: 0.22460858228328426
Fold 5 IBS: 0.21850495327987215
[I 2024-04-14 00:55:45,434] Trial 80 finished with value: 0.2166393324832414 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 6, 'max_depth': 16, 'n_estimators': 442, 'oob_score': True, 'max_samples': 0.7543724309332009, 'max_features': None, 'min_weight_fraction_leaf': 0.43164085935118673}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.1718240887458899
Fold 2 IBS: 0.191437443203446
Fold 3 IBS: 0.16491345446124533
Fold 4 IBS: 0.16398408770898562
Fold 5 IBS: 0.22656733476064983
[I 2024-04-14 00:55:56,114] Trial 81 finished with value: 0.18374528177604332 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 8, 'max_depth': 18, 'n_estimators': 355, 'oob_score': False, 'max_samples': 0.8723363913158229, 'max_features': None, 'min_weight_fraction_leaf': 

Fold 1 IBS: 0.18206223269633873
Fold 2 IBS: 0.18820585180072097
Fold 3 IBS: 0.17369714014143833
Fold 4 IBS: 0.1751220212365198
Fold 5 IBS: 0.22437209861320465
[I 2024-04-14 00:59:08,459] Trial 96 finished with value: 0.1886918688976445 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 8, 'n_estimators': 448, 'oob_score': False, 'max_samples': 0.8430407084230132, 'max_features': None, 'min_weight_fraction_leaf': 0.41419337591128474}. Best is trial 23 with value: 0.1829201586199189.
Fold 1 IBS: 0.17261810934600746
Fold 2 IBS: 0.19291443628076307
Fold 3 IBS: 0.16469348295311864
Fold 4 IBS: 0.16293280131529528
Fold 5 IBS: 0.2265701418039605
[I 2024-04-14 00:59:23,374] Trial 97 finished with value: 0.183945794339829 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 15, 'min_samples_leaf': 3, 'max_depth': 9, 'n_estimators': 499, 'oob_score': False, 'max_samples': 0.8784577990568653, 'max_features': None, 'min_weight_fraction_leaf': 0

In [149]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [150]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.826
train_ibs:  0.183


#### Test

In [151]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

In [152]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=19, max_features='log2', max_leaf_nodes=15,
                     max_samples=0.9995515317027949, min_samples_split=18,
                     min_weight_fraction_leaf=0.032175807966512454,
                     n_estimators=115, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.653


RandomSurvivalForest(max_depth=20, max_features=None, max_leaf_nodes=13,
                     max_samples=0.9248759152889626, min_samples_split=2,
                     min_weight_fraction_leaf=0.41894325026727597,
                     n_estimators=396, random_state=123)

test_ibs:  0.206


In [153]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [154]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [155]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 00:59:49,769] A new study created in memory with name: no-name-a9aa1914-577b-4f77-818c-8c1a81f29400


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7359307359307359
Fold 2 C-index: 0.8080357142857143
Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6948356807511737
[I 2024-04-14 00:59:52,275] Trial 0 finished with value: 0.7836129950691764 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7836129950691764.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:00:01,038] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 4 C-index: 0.8122362869198312
Fold 5 C-index: 0.6713615023474179
[I 2024-04-14 01:01:18,629] Trial 15 finished with value: 0.7684393821468035 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.7914452579440178.
Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.8013392857142857
Fold 3 C-index: 0.8333333333333334
Fold 4 C-index: 0.8248945147679325
Fold 5 C-index: 0.6267605633802817
[I 2024-04-14 01:01:22,286] Trial 16 finished with value: 0.77051229268592 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is 

Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.7924107142857143
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.6572769953051644
[I 2024-04-14 01:02:20,994] Trial 30 finished with value: 0.7671596620665804 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 11, 'min_samples_leaf': 12, 'max_depth': 5, 'n_estimators': 459, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.8382504072008882, 'min_weight_fraction_leaf': 0.23846812660748434}. Best is trial 12 with value: 0.7914452579440178.
Fold 1 C-index: 0.7229437229437229
Fold 2 C-index: 0.8169642857142857
Fold 3 C-index: 0.8529411764705882
Fold 4 C-index: 0.8248945147679325
Fold 5 C-index: 0.6384976525821596
[I 2024-04-14 01:02:28,897] Trial 31 finished with value: 0.7712482704957379 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 5, 'min_samples_leaf': 16, 'max_depth': 13, 'n_estimators': 500, 'oob_score': True, 'warm_start': True, 'max_features'

...

Fold 1 IBS: 0.16792251267623906
Fold 2 IBS: 0.2174484716230587
Fold 3 IBS: 0.16273781989489802
Fold 4 IBS: 0.1558940391227505
Fold 5 IBS: 0.23222422254692077
[I 2024-04-14 01:13:12,568] Trial 30 finished with value: 0.18724541317277338 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 6, 'min_samples_leaf': 2, 'max_depth': 12, 'n_estimators': 455, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.7293028765390641, 'min_weight_fraction_leaf': 0.08037045697723012}. Best is trial 23 with value: 0.1855727116611247.
Fold 1 IBS: 0.16646920043412164
Fold 2 IBS: 0.22144555868438975
Fold 3 IBS: 0.1612585234507463
Fold 4 IBS: 0.15276461377317607
Fold 5 IBS: 0.23760424924495455
[I 2024-04-14 01:13:18,750] Trial 31 finished with value: 0.18790842911747765 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 5, 'n_estimators': 209, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.83

Fold 1 IBS: 0.16512803721910052
Fold 2 IBS: 0.21893230349393553
Fold 3 IBS: 0.1593836385256823
Fold 4 IBS: 0.15058205502042205
Fold 5 IBS: 0.23336241815587846
[I 2024-04-14 01:14:56,009] Trial 45 finished with value: 0.18547769048300378 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 3, 'max_depth': 12, 'n_estimators': 276, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6931500413934508, 'min_weight_fraction_leaf': 0.021457257714753555}. Best is trial 32 with value: 0.18528124104479504.
Fold 1 IBS: 0.1659546836323674
Fold 2 IBS: 0.2153197227930472
Fold 3 IBS: 0.16360867863772716
Fold 4 IBS: 0.15557836846786757
Fold 5 IBS: 0.233640633124344
[I 2024-04-14 01:15:03,832] Trial 46 finished with value: 0.18682041733107066 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 13, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 283, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0

Fold 1 IBS: 0.17342492293804432
Fold 2 IBS: 0.2110732537830341
Fold 3 IBS: 0.16831819527481992
Fold 4 IBS: 0.16873369129336893
Fold 5 IBS: 0.22655614890084055
[I 2024-04-14 01:17:18,245] Trial 60 finished with value: 0.18962124243802156 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 4, 'min_samples_leaf': 5, 'max_depth': 14, 'n_estimators': 439, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.5735294094022945, 'min_weight_fraction_leaf': 0.10077092974890593}. Best is trial 53 with value: 0.18257594357755064.
Fold 1 IBS: 0.16275315104094723
Fold 2 IBS: 0.22028966274589679
Fold 3 IBS: 0.16253743592400494
Fold 4 IBS: 0.15352895984062664
Fold 5 IBS: 0.23021227496706434
[I 2024-04-14 01:17:29,070] Trial 61 finished with value: 0.18586429690370798 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 4, 'min_samples_leaf': 3, 'max_depth': 11, 'n_estimators': 412, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples':

Fold 1 IBS: 0.16278536069617075
Fold 2 IBS: 0.22061911570320727
Fold 3 IBS: 0.15948095174532076
Fold 4 IBS: 0.1500453215492157
Fold 5 IBS: 0.23108140998200039
[I 2024-04-14 01:20:05,452] Trial 75 finished with value: 0.18480243193518295 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 463, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.6487943567829829, 'min_weight_fraction_leaf': 0.011079149496446542}. Best is trial 53 with value: 0.18257594357755064.
Fold 1 IBS: 0.1649931201311961
Fold 2 IBS: 0.2191098546301487
Fold 3 IBS: 0.1604180424655896
Fold 4 IBS: 0.15133070246635846
Fold 5 IBS: 0.23270813963892614
[I 2024-04-14 01:20:20,817] Trial 76 finished with value: 0.18571197186644378 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 496, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0

Fold 1 IBS: 0.18162653052097805
Fold 2 IBS: 0.20612651784753333
Fold 3 IBS: 0.1771784733434652
Fold 4 IBS: 0.18941755917999278
Fold 5 IBS: 0.21614311489575638
[I 2024-04-14 01:22:45,824] Trial 90 finished with value: 0.19409843915754516 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 6, 'min_samples_leaf': 1, 'max_depth': 20, 'n_estimators': 362, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.7635642567316503, 'min_weight_fraction_leaf': 0.280081595660318}. Best is trial 53 with value: 0.18257594357755064.
Fold 1 IBS: 0.16186089564069253
Fold 2 IBS: 0.21899259491528042
Fold 3 IBS: 0.15921485444738478
Fold 4 IBS: 0.148193042465612
Fold 5 IBS: 0.23190417972474658
[I 2024-04-14 01:22:55,857] Trial 91 finished with value: 0.18403311343874326 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 19, 'n_estimators': 323, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.789

In [156]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [157]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.794
train_ibs:  0.183


#### Test

In [158]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [159]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=12, max_features=None, max_leaf_nodes=6,
                   max_samples=0.7030590704740037, min_samples_leaf=6,
                   min_samples_split=13,
                   min_weight_fraction_leaf=0.020016091327628317,
                   n_estimators=273, random_state=123, warm_start=True)

C-index score: 0.618


ExtraSurvivalTrees(max_depth=13, max_features='log2', max_leaf_nodes=6,
                   max_samples=0.7048480399273186, min_samples_leaf=1,
                   min_samples_split=12,
                   min_weight_fraction_leaf=0.0010291888027469595,
                   n_estimators=485, random_state=123, warm_start=True)

IBS: 0.215


In [160]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [161]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 01:24:24,685] A new study created in memory with name: no-name-ef53575b-01ef-4bf9-9c9a-82bb9c4c32ef


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:25:13,700] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:25:41,115] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:39:22,377] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7334874350516556.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:41:00,557] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 01:57:54,269] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7334874350516556.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 01:59:50,361] Trial 26 finished with value: 0.5421550134138162 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446,

...

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 05:24:13,848] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9027683411994925, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.13472529918097312, 'n_estimators': 381, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.36938177041618503, 'max_features': 'auto', 'min_impurity_decrease': 1.1016843774656315e-07, 'validation_fraction': 0.898549324711475, 'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 05:26:18,038] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.8324789538052517, 'learning_rate': 0.0985092209

Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-14 05:51:19,184] Trial 66 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9104313304151431, 'learning_rate': 0.014815622348446418, 'dropout_rate': 0.1415896279444537, 'n_estimators': 149, 'criterion': 'squared_error', 'ccp_alpha': 0.7060910424624821, 'min_weight_fraction_leaf': 0.2796836959018326, 'max_features': 'auto', 'min_impurity_decrease': 5.127880425748818e-06, 'validation_fraction': 0.916501041728546, 'min_samples_split': 19, 'max_leaf_nodes': 16, 'min_samples_leaf': 18, 'max_depth': 5}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 05:53:43,709] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.802369464636257, 'learning_rate': 0.0041755238500

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 06:22:57,049] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8921532985730489, 'learning_rate': 0.021931279910218017, 'dropout_rate': 0.19284931428290142, 'n_estimators': 490, 'criterion': 'squared_error', 'ccp_alpha': 0.5905545096948588, 'min_weight_fraction_leaf': 0.07057220045251793, 'max_features': 'log2', 'min_impurity_decrease': 1.637559261311236e-05, 'validation_fraction': 0.8150762480319586, 'min_samples_split': 5, 'max_leaf_nodes': 13, 'min_samples_leaf': 8, 'max_depth': 5}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-14 06:25:36,171] Trial 78 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.7443996478445325, 'learning_rate': 0.086932868663

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 06:50:32,413] Trial 88 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9447987915285996, 'learning_rate': 0.014159070275535094, 'dropout_rate': 0.18102387516231752, 'n_estimators': 470, 'criterion': 'squared_error', 'ccp_alpha': 0.5935000311483954, 'min_weight_fraction_leaf': 0.2151508441811604, 'max_features': 'auto', 'min_impurity_decrease': 3.72786238165322e-07, 'validation_fraction': 0.898328514849679, 'min_samples_split': 18, 'max_leaf_nodes': 18, 'min_samples_leaf': 12, 'max_depth': 2}. Best is trial 53 with value: 0.2150432178150902.
Fold 1 IBS: 0.21292314886383587
Fold 2 IBS: 0.2213935582997429
Fold 3 IBS: 0.20349492943164285
Fold 4 IBS: 0.2234443788901035
Fold 5 IBS: 0.2178743294294726
[I 2024-04-14 06:52:01,820] Trial 89 finished with value: 0.21582606898295956 and parameters: {'subsample': 0.9777744983689288, 'learning_rate': 0.010383571834622

Fold 3 IBS: 0.20237857642828805
Fold 4 IBS: 0.22105646622231043
Fold 5 IBS: 0.21749535106188492
[I 2024-04-14 07:05:00,203] Trial 99 finished with value: 0.21481876339222578 and parameters: {'subsample': 0.8569271041566342, 'learning_rate': 0.01377474153403631, 'dropout_rate': 0.2044520049398774, 'n_estimators': 378, 'criterion': 'squared_error', 'ccp_alpha': 0.004422201299274769, 'min_weight_fraction_leaf': 0.13940114558509004, 'max_features': 'auto', 'min_impurity_decrease': 0.00015837847703956763, 'validation_fraction': 0.9580200141092102, 'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 13, 'max_depth': 2}. Best is trial 99 with value: 0.21481876339222578.


* Best trial for IBS: 
 FrozenTrial(number=99, state=TrialState.COMPLETE, values=[0.21481876339222578], datetime_start=datetime.datetime(2024, 4, 14, 7, 3, 54, 536311), datetime_complete=datetime.datetime(2024, 4, 14, 7, 5, 0, 201656), params={'subsample': 0.8569271041566342, 'learning_rate': 0.013774741534036

In [162]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [163]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.745
train_ibs:  0.215


#### Test

In [164]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [165]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.03063070291051248,
                                 criterion='squared_error',
                                 dropout_rate=0.2576884847115747,
                                 learning_rate=0.007359366951045268,
                                 max_depth=16, max_features=1,
                                 max_leaf_nodes=16,
                                 min_impurity_decrease=1.3903906488794697e-07,
                                 min_samples_leaf=14, min_samples_split=20,
                                 min_weight_fraction_leaf=0.38180626899357545,
                                 n_estimators=96, random_state=123,
                                 subsample=0.8938290428827321,
                                 validation_fraction=0.9577535215137098)

C-index score: 0.603


GradientBoostingSurvivalAnalysis(ccp_alpha=0.004422201299274769,
                                 criterion='squared_error',
                                 dropout_rate=0.2044520049398774,
                                 learning_rate=0.01377474153403631, max_depth=2,
                                 max_features='auto', max_leaf_nodes=17,
                                 min_impurity_decrease=0.00015837847703956763,
                                 min_samples_leaf=13, min_samples_split=13,
                                 min_weight_fraction_leaf=0.13940114558509004,
                                 n_estimators=378, random_state=123,
                                 subsample=0.8569271041566342,
                                 validation_fraction=0.9580200141092102)

IBS: 0.22


In [166]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [167]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [168]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 07:05:14,604] A new study created in memory with name: no-name-e08a3be9-5624-483e-bc3a-7c0f4905ce7c


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.6473214285714286
Fold 3 C-index: 0.7230392156862745
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.5985915492957746
[I 2024-04-14 07:05:16,072] Trial 0 finished with value: 0.70555546693142 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.70555546693142.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.6339285714285714
Fold 3 C-index: 0.7352941176470589
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.5985915492957746
[I 2024-04-14 07:05:26,193] Trial 1 finished with value: 0.7053278758950055 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.70555546693142.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.6473214285714286
Fold 3 C-index: 0.7401960784313726
Fold 4 C-in

Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.65625
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.5868544600938967
[I 2024-04-14 07:06:27,688] Trial 19 finished with value: 0.7222412032042431 and parameters: {'subsample': 0.16691848936207518, 'dropout_rate': 0.8098474688245095, 'n_estimators': 341, 'learning_rate': 0.03936102119031181}. Best is trial 16 with value: 0.7359390054989564.
Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.6517857142857143
Fold 3 C-index: 0.7573529411764706
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.5821596244131455
[I 2024-04-14 07:06:30,923] Trial 20 finished with value: 0.716885334094415 and parameters: {'subsample': 0.29109263215046977, 'dropout_rate': 0.5903532280380733, 'n_estimators': 258, 'learning_rate': 0.022816282680087063}. Best is trial 16 with value: 0.7359390054989564.
Fold 1 C-index: 0.7705627705627706
Fold 2 C-index: 0.6785714285714286
Fold 3 C-index: 0.7843137254901961
Fold 4 C-i

Fold 1 C-index: 0.7705627705627706
Fold 2 C-index: 0.6741071428571429
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.8227848101265823
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 07:07:46,748] Trial 38 finished with value: 0.7324194175068689 and parameters: {'subsample': 0.10130616072821844, 'dropout_rate': 0.2860710109197408, 'n_estimators': 229, 'learning_rate': 0.028067533175364198}. Best is trial 22 with value: 0.7368779726351067.
Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.6607142857142857
Fold 3 C-index: 0.7745098039215687
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.5774647887323944
[I 2024-04-14 07:07:49,438] Trial 39 finished with value: 0.7211634537929987 and parameters: {'subsample': 0.21652307168545082, 'dropout_rate': 0.904275906561924, 'n_estimators': 309, 'learning_rate': 0.011375143101675419}. Best is trial 22 with value: 0.7368779726351067.
Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.6428571428571429
Fold 3 C-index: 0.747549019607843

...

Fold 1 IBS: 0.19426171832065434
Fold 2 IBS: 0.19715811856127513
Fold 3 IBS: 0.17946469994393788
Fold 4 IBS: 0.20156135236511127
Fold 5 IBS: 0.21995157160040674
[I 2024-04-14 07:10:37,606] Trial 20 finished with value: 0.1984794921582771 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.19556291407955367.
Fold 1 IBS: 0.1974689136933835
Fold 2 IBS: 0.20116625589103407
Fold 3 IBS: 0.18307602213830743
Fold 4 IBS: 0.20415973589751046
Fold 5 IBS: 0.21825141892231678
[I 2024-04-14 07:10:37,760] Trial 21 finished with value: 0.20082446930851044 and parameters: {'subsample': 0.11437919910307487, 'dropout_rate': 0.9754161103921428, 'n_estimators': 33, 'learning_rate': 0.042594274103339054}. Best is trial 19 with value: 0.19556291407955367.
Fold 1 IBS: 0.2131782018406857
Fold 2 IBS: 0.21966297563854562
Fold 3 IBS: 0.20310325082392688
Fold 4 IBS: 0.22331313612323705
Fold 5 

Fold 4 IBS: 0.19062854819614738
Fold 5 IBS: 0.2531160401238152
[I 2024-04-14 07:10:45,384] Trial 39 finished with value: 0.19497583360857934 and parameters: {'subsample': 0.2485509072011421, 'dropout_rate': 0.2538651184588351, 'n_estimators': 81, 'learning_rate': 0.05575307634287385}. Best is trial 32 with value: 0.19059504692275583.
Fold 1 IBS: 0.1742564619506075
Fold 2 IBS: 0.2618544585408401
Fold 3 IBS: 0.1954849738012067
Fold 4 IBS: 0.2706753920831236
Fold 5 IBS: 0.29857369751177665
[I 2024-04-14 07:10:45,994] Trial 40 finished with value: 0.2401689967775109 and parameters: {'subsample': 0.24862688656216192, 'dropout_rate': 0.1585668555692103, 'n_estimators': 167, 'learning_rate': 0.08018822574072365}. Best is trial 32 with value: 0.19059504692275583.
Fold 1 IBS: 0.17431035295693673
Fold 2 IBS: 0.19126346272609226
Fold 3 IBS: 0.16576671229380383
Fold 4 IBS: 0.1869585355531429
Fold 5 IBS: 0.24797577822458722
[I 2024-04-14 07:10:46,280] Trial 41 finished with value: 0.193254968350912

Fold 4 IBS: 0.18579562086352025
Fold 5 IBS: 0.25844938334040174
[I 2024-04-14 07:10:54,528] Trial 58 finished with value: 0.19562807510826627 and parameters: {'subsample': 0.17455703880609846, 'dropout_rate': 0.5681825165583033, 'n_estimators': 177, 'learning_rate': 0.02857135261777332}. Best is trial 57 with value: 0.19046594953507473.
Fold 1 IBS: 0.16918296299612848
Fold 2 IBS: 0.19385471607741084
Fold 3 IBS: 0.16423780215620198
Fold 4 IBS: 0.17037260513956887
Fold 5 IBS: 0.2606061379815798
[I 2024-04-14 07:10:55,486] Trial 59 finished with value: 0.191650844870178 and parameters: {'subsample': 0.10169393529521677, 'dropout_rate': 0.4915957741107889, 'n_estimators': 156, 'learning_rate': 0.03590100641045702}. Best is trial 57 with value: 0.19046594953507473.
Fold 1 IBS: 0.18520166379023767
Fold 2 IBS: 0.19128134476499634
Fold 3 IBS: 0.16985515532232098
Fold 4 IBS: 0.1930586229948487
Fold 5 IBS: 0.22818293335247655
[I 2024-04-14 07:10:56,171] Trial 60 finished with value: 0.1935159440

Fold 2 IBS: 0.19368370154067086
Fold 3 IBS: 0.17466480787356398
Fold 4 IBS: 0.19441480400408118
Fold 5 IBS: 0.22030789644181628
[I 2024-04-14 07:11:05,321] Trial 78 finished with value: 0.19457534362611795 and parameters: {'subsample': 0.1289388075828276, 'dropout_rate': 0.3540380121430278, 'n_estimators': 45, 'learning_rate': 0.05031324288681145}. Best is trial 57 with value: 0.19046594953507473.
Fold 1 IBS: 0.19996995210195043
Fold 2 IBS: 0.20487136825192176
Fold 3 IBS: 0.1868270122179074
Fold 4 IBS: 0.2094329875633396
Fold 5 IBS: 0.2148277651809021
[I 2024-04-14 07:11:05,441] Trial 79 finished with value: 0.2031858170632043 and parameters: {'subsample': 0.14605546351673254, 'dropout_rate': 0.8543708031677456, 'n_estimators': 11, 'learning_rate': 0.0983514933405076}. Best is trial 57 with value: 0.19046594953507473.
Fold 1 IBS: 0.17164249487743055
Fold 2 IBS: 0.19348710238254757
Fold 3 IBS: 0.16672720255994614
Fold 4 IBS: 0.1811824773100978
Fold 5 IBS: 0.2528607570353557
[I 2024-04-1

Fold 4 IBS: 0.1725939047574419
Fold 5 IBS: 0.26916428963724165
[I 2024-04-14 07:11:13,907] Trial 97 finished with value: 0.19488610025674777 and parameters: {'subsample': 0.11658895674183224, 'dropout_rate': 0.5693703139031614, 'n_estimators': 116, 'learning_rate': 0.05468275806027857}. Best is trial 57 with value: 0.19046594953507473.
Fold 1 IBS: 0.1819200561017552
Fold 2 IBS: 0.19484138963367623
Fold 3 IBS: 0.17803463882277085
Fold 4 IBS: 0.20400784936010083
Fold 5 IBS: 0.24720801744754423
[I 2024-04-14 07:11:14,356] Trial 98 finished with value: 0.2012023902731695 and parameters: {'subsample': 0.6256495106817341, 'dropout_rate': 0.5171615627018341, 'n_estimators': 140, 'learning_rate': 0.02311141545634017}. Best is trial 57 with value: 0.19046594953507473.
Fold 1 IBS: 0.18497690412089743
Fold 2 IBS: 0.2047865657788103
Fold 3 IBS: 0.1778718387416613
Fold 4 IBS: 0.21398164113622073
Fold 5 IBS: 0.24576001387482957
[I 2024-04-14 07:11:14,565] Trial 99 finished with value: 0.205475392730

In [169]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [170]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.737
train_ibs:  0.19


#### Test

In [171]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [172]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.9771912810322235,
                                              learning_rate=0.012269705471982865,
                                              n_estimators=419,
                                              random_state=123,
                                              subsample=0.10322253994980501)

C-index score: 0.509


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.47900572112881107,
                                              learning_rate=0.028234629788730554,
                                              n_estimators=177,
                                              random_state=123,
                                              subsample=0.10158446690685444)

IBS: 0.259


In [173]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [174]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.826,1.0
ExtraSurvivalTrees,0.794,2.0
CoxLasso,0.746,3.5
CoxElastic,0.746,3.5
GradientBoosting,0.745,5.0
CoxPH,0.743,6.0
ComponentwiseGradientBoosting,0.737,7.0
CoxRidge,0.675,8.0


In [175]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.183,1.5
ExtraSurvivalTrees,0.183,1.5
CoxLasso,0.189,3.5
CoxElastic,0.189,3.5
CoxPH,0.190,5.5
ComponentwiseGradientBoosting,0.190,5.5
GradientBoosting,0.215,7.0
CoxRidge,0.217,8.0


In [176]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.653,1.0
ExtraSurvivalTrees,0.618,2.0
GradientBoosting,0.603,3.0
CoxLasso,0.592,4.5
CoxElastic,0.592,4.5
CoxPH,0.581,6.0
CoxRidge,0.534,7.0
ComponentwiseGradientBoosting,0.509,8.0


In [177]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
Randomsurvivalforest,0.206,1.0
ExtraSurvivalTrees,0.215,2.0
GradientBoosting,0.220,3.0
CoxRidge,0.221,4.0
ComponentwiseGradientBoosting,0.259,5.0
CoxLasso,0.280,6.5
CoxElastic,0.280,6.5
CoxPH,0.281,8.0


In [178]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/os/yeojohnson/no_selection/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_os_yeojohnson_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [179]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-14
